In [ ]:
import json
import os
import sys
from IPython.display import Markdown, display, update_display
from openai import OpenAI
sys.path.append(os.path.abspath(".."))
from scraper import scrap_website, scrap_website_links

In [7]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2"

In [14]:
links = scrap_website_links("https://www.udemy.com")
links

[]

In [15]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [16]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = scrap_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [17]:
print(get_links_user_prompt("https://www.udemy.com"))


Here is the list of links on the website https://www.udemy.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/cart/
/mobile/
/invite/
/support/
/udemy-business/?ref=ub_header_home&locale=en_US
/udemy-business/plans/?ref=ufb_header_plans&locale=en_US
/udemy-business/request-demo-mx/?ref=ufb_header_demo&locale=en_US
#main-content-anchor
/
/browse/certification/
/browse/certification/comptia-certifications/
/browse/certification/aws-certifications/
/browse/certification/project-management-institute-pmi-certifications/
/featured-topics/
/udemy-business/?locale=en_US&path=request-demo-mx%2F&ref=footer-ad
/


In [18]:
def select_relevant_links(url):
    openai = OpenAI(base_url = OLLAMA_BASE_URL, api_key="ollama")
    response = openai.chat.completions.create(
        model = MODEL,
        messages = [
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [50]:
select_relevant_links("https://www.udemy.com")

{'links': [{'type': 'about page', 'url': 'https://www.udemy.com/about'},
  {'type': 'company page', 'url': 'https://www.udemy.com/about'},
  {'type': 'features page', 'url': 'https://www.udemy.com/featured-topics'},
  {'type': 'careers/jobs page', 'url': 'https://www.udemy.com/careers'}]}

### TO MAKE BROCHURE

In [19]:
def fetch_page_and_all_relevant_links(url):
    contents = scrap_website(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += scrap_website(link["url"])
    return result

In [20]:
print(fetch_page_and_all_relevant_links("https://www.udemy.com"))

## Landing Page:

Udemy: Online Courses for Skills, Careers & AI

Search bar
Search for anything
Site navigation
Most popular
More from Udemy
Udemy Business
Get the app
Invite friends
Help and Support
English
Menu
About Udemy Business
Compare Plans
Try Udemy Business
Skip to content
Categories
Search for anything
Show more
Show less
Get certified and get ahead in your career
Prep for certifications with comprehensive courses, practice tests, and special offers on exam vouchers.
Explore certifications and vouchers
CompTIA
Cloud, Networking, Cybersecurity
AWS
Cloud, AI, Coding, Networking
PMI
Project & Program Management
Popular Skills
Show all trending skills
Top companies choose
Udemy Business
to build in-demand career skills.
© 2026 Udemy, Inc.
English
## Relevant Links:


### Link: about page
Udemy: Online Courses for Skills, Careers & AI

Search bar
Search for anything
Site navigation
Most popular
More from Udemy
Udemy Business
Get the app
Invite friends
Help and Support
English
Men

In [21]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [22]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [23]:
get_brochure_user_prompt("Udemy", "https://www.udemy.com")

'\nYou are looking at a company called: Udemy\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nUdemy: Online Courses for Skills, Careers & AI\n\nSearch bar\nSearch for anything\nSite navigation\nMost popular\nMore from Udemy\nUdemy Business\nGet the app\nInvite friends\nHelp and Support\nEnglish\nMenu\nAbout Udemy Business\nCompare Plans\nTry Udemy Business\nSkip to content\nCategories\nSearch for anything\nShow more\nShow less\nGet certified and get ahead in your career\nPrep for certifications with comprehensive courses, practice tests, and special offers on exam vouchers.\nExplore certifications and vouchers\nCompTIA\nCloud, Networking, Cybersecurity\nAWS\nCloud, AI, Coding, Networking\nPMI\nProject & Program Management\nPopular Skills\nShow all trending skills\nTop companies choose\nUdemy Business\nto build in-demand career skills.\n© 2026 Ude

In [24]:
def create_brochure(company_name, url):
    openai = OpenAI(base_url = OLLAMA_BASE_URL, api_key="ollama")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [25]:
create_brochure("Udemy", "https://www.udemy.com")

# The Udemy Way: Learning, Growing, and Thriving Together

## Our Story

Udemy is more than just an online learning platform - it's a community of curious minds and passionate learners. Founded on the principle that learning should be accessible to everyone, we strive to empower individuals by providing comprehensive courses, expert instructors, and a supportive environment for growth.

## Who We Are

Our customers come from diverse backgrounds and industries, seeking to upskill, reskill, or simply upgrade their knowledge. With millions of students enrolled on our platform every year, we're committed to delivering the highest quality learning experiences tailored to their needs.

From IT professionals looking to enhance their cloud computing skills to entrepreneurs wanting to boost their business acumen, our courses cater to diverse interests and career goals.

## Our Culture

At Udemy, we value:

*   **Flexibility**: Learning should be on your schedule, not dictated by it. We offer self-paced courses so you can learn whenever, wherever.
*   **Expertise**: Our instructors are industry thought leaders who share their knowledge to help students achieve real-world results.
*   **Community**: Join our vibrant community of learners and educators who collaborate, share ideas, and inspire each other.

## Jobs at Udemy

If you're passionate about learning, teaching, or innovating in tech, consider joining our team. We offer a range of roles from customer support to content creation, software engineering to marketing - there's something for everyone!

**What we look for:**

*   Passion for education and lifelong learning
*   Excellent communication and collaboration skills
*   Innovation mindset (we love thinking outside the box!)
*   Commitment to our values (flexibility, expertise, community)

Ready to explore job opportunities?

## Join the Movement

Get certified, get ahead in your career, or learn a new skill - the possibilities are endless. Invest in yourself and experience the Udemy difference.

---

**Find courses that fit you**

Browse by topic, skill level, or industry:

*   CompTIA
*   Cloud
*   Networking
*   Cybersecurity (compiling on AWS)
*   AI/Cloud/Coding/Networking

Visit our course catalog and find top companies choosing Udemy to build in-demand career skills.

**Find a role that fits your dreams**

From customer support to software engineering, consider our job opportunities today:

1.  Explore our Careers page for the latest openings
2.  Follow us on social media for updates
3.  Register with us for future jobs and be the first to know about new roles.